# Day 18 — SQL Fundamentals Exercises

In this notebook, we practice querying data where it lives using **DuckDB**. We'll start by generating a mock e-commerce database, and then solve 5 exercises covering standard SQL capabilities.

### 1. Database Setup
We generate mock data for `customers`, `products`, `orders`, and `order_items` and register them as tables in our in-memory DuckDB connection.

In [4]:
import pandas as pd
import numpy as np
import duckdb

# Set seed for reproducibility
np.random.seed(42)

# 1. Customers Table
num_customers = 50
customers_df = pd.DataFrame({
    'customer_id': range(1, num_customers + 1),
    'name': [f"Customer {i}" for i in range(1, num_customers + 1)],
    'country': np.random.choice(['USA', 'Canada', 'UK', 'Germany', 'France'], num_customers),
    'signup_date': pd.date_range(start='2024-01-01', periods=num_customers, freq='D').strftime('%Y-%m-%d')
})

# 2. Products Table
products_df = pd.DataFrame({
    'product_id': range(1, 11),
    'name': ['Laptop', 'Smartphone', 'Headphones', 'Monitor', 'Keyboard', 
             'Desk Lamp', 'Office Chair', 'Notebook', 'Coffee Mug', 'Backpack'],
    'category': ['Electronics', 'Electronics', 'Electronics', 'Electronics', 'Electronics', 
                 'Furniture', 'Furniture', 'Stationery', 'Stationery', 'Accessories'],
    'price': [1200.0, 800.0, 150.0, 300.0, 80.0, 45.0, 250.0, 5.0, 12.0, 60.0]
})

# 3. Orders Table
num_orders = 120
orders_df = pd.DataFrame({
    'order_id': range(1, num_orders + 1),
    'customer_id': np.random.choice(customers_df['customer_id'], num_orders),
    'order_date': pd.date_range(start='2024-02-01', periods=num_orders, freq='12h').strftime('%Y-%m-%d')
})

# 4. Order Items Table
order_items_list = []
item_id_counter = 1
for order_id in orders_df['order_id']:
    num_items = np.random.randint(1, 5)
    selected_product_ids = np.random.choice(products_df['product_id'], num_items, replace=False)
    for p_id in selected_product_ids:
        price = products_df.loc[products_df['product_id'] == p_id, 'price'].values[0]
        qty = int(np.random.choice([1, 2, 3]))
        order_items_list.append({
            'item_id': item_id_counter,
            'order_id': order_id,
            'product_id': p_id,
            'quantity': qty,
            'price': price
        })
        item_id_counter += 1

order_items_df = pd.DataFrame(order_items_list)

# Compute actual total_amount for orders to maintain database integrity
order_totals = order_items_df.groupby('order_id').apply(lambda x: (x['quantity'] * x['price']).sum(), include_groups=False).reset_index(name='total_amount')
orders_df = orders_df.merge(order_totals, on='order_id', how='left')

# Initialize DuckDB connection and register DataFrames
con = duckdb.connect(database=':memory:')
con.register('customers', customers_df)
con.register('products', products_df)
con.register('orders', orders_df)
con.register('order_items', order_items_df)

print("Mock database successfully generated and registered as tables: customers, products, orders, order_items!")
# Show schema info
con.sql("SHOW TABLES;").show()

Mock database successfully generated and registered as tables: customers, products, orders, order_items!
┌─────────────┐
│    name     │
│   varchar   │
├─────────────┤
│ customers   │
│ order_items │
│ orders      │
│ products    │
└─────────────┘



### 2. Exercises
Complete the following exercises using DuckDB SQL. Solutions are executed directly below each query definition.

#### Exercise 1: SELECT/WHERE
Find all products in the 'Electronics' category with a price greater than $100.

In [2]:
ex1_query = """
SELECT *
FROM products
WHERE category = 'Electronics' AND price > 100;
"""
con.sql(ex1_query).show()

┌────────────┬────────────┬─────────────┬────────┐
│ product_id │    name    │  category   │ price  │
│   int64    │  varchar   │   varchar   │ double │
├────────────┼────────────┼─────────────┼────────┤
│          1 │ Laptop     │ Electronics │ 1200.0 │
│          2 │ Smartphone │ Electronics │  800.0 │
│          3 │ Headphones │ Electronics │  150.0 │
│          4 │ Monitor    │ Electronics │  300.0 │
└────────────┴────────────┴─────────────┴────────┘



#### Exercise 2: GROUP BY/HAVING
Calculate total orders and total revenue (sum of total_amount) per customer, showing only those customers who spent more than $1500 total.

In [3]:
ex2_query = """
SELECT customer_id, COUNT(order_id) as total_orders, SUM(total_amount) as total_spent
FROM orders
GROUP BY customer_id
HAVING SUM(total_amount) > 1500
ORDER BY total_spent DESC;
"""
con.sql(ex2_query).show()

┌─────────────┬──────────────┬─────────────┐
│ customer_id │ total_orders │ total_spent │
│    int64    │    int64     │   double    │
├─────────────┼──────────────┼─────────────┤
│          14 │            5 │     11879.0 │
│           7 │            4 │     10387.0 │
│          47 │            4 │      8881.0 │
│          44 │            4 │      8635.0 │
│          39 │            4 │      8191.0 │
│          42 │            4 │      8055.0 │
│          35 │            6 │      7540.0 │
│          37 │            3 │      7230.0 │
│          26 │            4 │      6945.0 │
│          36 │            3 │      6119.0 │
│           · │            · │         ·   │
│           · │            · │         ·   │
│           · │            · │         ·   │
│          22 │            2 │      2760.0 │
│          25 │            2 │      2700.0 │
│          50 │            1 │      2686.0 │
│          16 │            1 │      2635.0 │
│          24 │            4 │      2616.0 │
│         

#### Exercise 3: JOINs
Retrieve the order_id, order_date, customer name, and the product name and quantity purchased for the first 10 items in the database.

In [4]:
ex3_query = """
SELECT oi.order_id, o.order_date, c.name as customer_name, p.name as product_name, oi.quantity
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON oi.product_id = p.product_id
LIMIT 10;
"""
con.sql(ex3_query).show()

┌──────────┬────────────┬───────────────┬──────────────┬──────────┐
│ order_id │ order_date │ customer_name │ product_name │ quantity │
│  int64   │  varchar   │    varchar    │   varchar    │  int64   │
├──────────┼────────────┼───────────────┼──────────────┼──────────┤
│        1 │ 2024-02-01 │ Customer 28   │ Laptop       │        3 │
│        2 │ 2024-02-01 │ Customer 47   │ Smartphone   │        2 │
│        2 │ 2024-02-01 │ Customer 47   │ Headphones   │        1 │
│        2 │ 2024-02-01 │ Customer 47   │ Notebook     │        1 │
│        2 │ 2024-02-01 │ Customer 47   │ Keyboard     │        1 │
│        3 │ 2024-02-02 │ Customer 7    │ Monitor      │        3 │
│        3 │ 2024-02-02 │ Customer 7    │ Smartphone   │        3 │
│        3 │ 2024-02-02 │ Customer 7    │ Backpack     │        2 │
│        4 │ 2024-02-02 │ Customer 44   │ Headphones   │        2 │
│        4 │ 2024-02-02 │ Customer 44   │ Keyboard     │        3 │
└──────────┴────────────┴───────────────┴───────

#### Exercise 4: Subqueries & CTEs
Find all customers who have placed orders that are larger than the overall average order total. We'll use a Common Table Expression (CTE) for clarity.

In [5]:
ex4_query = """
WITH avg_order AS (
    SELECT AVG(total_amount) as avg_amt FROM orders
)
SELECT DISTINCT c.customer_id, c.name, o.order_id, o.total_amount
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.total_amount > (SELECT avg_amt FROM avg_order)
ORDER BY o.total_amount DESC;
"""
con.sql(ex4_query).show()

┌─────────────┬─────────────┬──────────┬──────────────┐
│ customer_id │    name     │ order_id │ total_amount │
│    int64    │   varchar   │  int64   │    double    │
├─────────────┼─────────────┼──────────┼──────────────┤
│          44 │ Customer 44 │       77 │       5480.0 │
│          36 │ Customer 36 │       24 │       5290.0 │
│          27 │ Customer 27 │       80 │       4800.0 │
│          42 │ Customer 42 │       98 │       4700.0 │
│           6 │ Customer 6  │       15 │       4500.0 │
│          39 │ Customer 39 │      108 │       4386.0 │
│           7 │ Customer 7  │       48 │       4312.0 │
│           1 │ Customer 1  │       46 │       3950.0 │
│          18 │ Customer 18 │       19 │       3910.0 │
│          14 │ Customer 14 │       86 │       3900.0 │
│           · │      ·      │        · │          ·   │
│           · │      ·      │        · │          ·   │
│           · │      ·      │        · │          ·   │
│          29 │ Customer 29 │       43 │       1

#### Exercise 5: Window Functions
Rank the products by price within each category, and show their relative rank.

In [6]:
ex5_query = """
SELECT name, category, price,
       RANK() OVER(PARTITION BY category ORDER BY price DESC) as price_rank
FROM products;
"""
con.sql(ex5_query).show()

┌──────────────┬─────────────┬────────┬────────────┐
│     name     │  category   │ price  │ price_rank │
│   varchar    │   varchar   │ double │   int64    │
├──────────────┼─────────────┼────────┼────────────┤
│ Backpack     │ Accessories │   60.0 │          1 │
│ Laptop       │ Electronics │ 1200.0 │          1 │
│ Smartphone   │ Electronics │  800.0 │          2 │
│ Monitor      │ Electronics │  300.0 │          3 │
│ Headphones   │ Electronics │  150.0 │          4 │
│ Keyboard     │ Electronics │   80.0 │          5 │
│ Office Chair │ Furniture   │  250.0 │          1 │
│ Desk Lamp    │ Furniture   │   45.0 │          2 │
│ Coffee Mug   │ Stationery  │   12.0 │          1 │
│ Notebook     │ Stationery  │    5.0 │          2 │
└──────────────┴─────────────┴────────┴────────────┘
  10 rows                                4 columns

